<a href="https://colab.research.google.com/github/DanilaKrug/ai_miit/blob/practice-4/notebooks/04_dirty_orders_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика 4. Грязный Excel и оркестратор: субагенты, код и ограничители

**Дисциплина:** Генеративный ИИ и ИИ-агенты для транспортной логистики (РУТ (МИИТ), магистратура)
**Время:** 60–90 минут. **Среда:** Google Colab.

## Что мы сегодня делаем

Пришёл Excel из трёх листов: шапка на одном листе с шестой строки, на другом колонки называются
иначе, на третьем заголовков нет вовсе. Внутри — 120 заявок на подачу вагонов, перемешанных с
«ИТОГО», дублями, опечатками и датами в четырёх форматах. Нужно загрузить заявки в базу и ответить
по ним на вопрос агента.

Модель сегодня **не видит ни одной строки данных целиком**. Она видит десять строк листа и
отвечает: где шапка и что значит каждая колонка. Всё остальное делает ваш код.

| TODO | Что вы пишете | Что делает «Проверка» |
|---|---|---|
| **1** | `make_subagent(sheet)` — четыре поля субагента: инструкция, контекст, инструменты, модель | оркестратор запускает три субагента; их раскладки листов сверяются с эталоном |
| **2** | `normalize(rows, layout)` — пропуск пустых и «ИТОГО», даты, количество, дубли | 120 уникальных заявок, даты ISO, количество `int` или `None` |
| **3** | `match_station(name, known)` — станция из справочника, а не догадка | 4 опечатки исправлены, верные станции не тронуты |
| **4** | `check_limits(state, sheet)` — ограничители оркестратора | лишний вызов получает текст ошибки, у субагентов нет `spawn_agent` |

У каждого задания одинаковая форма: зачем, что написать, ячейка с заданием и ячейка
**«Проверка TODO N»**. Готовые ячейки свёрнуты: видны заголовок и результат, код раскрывается по щелчку.

**Критерий сдачи:** `check()` в конце — **6 из 6**. SQL пишет код: в аргументах инструмента
`query_orders` SQL быть не может. Минимум на занятии — проверки 1–3 (TODO 1–2),
TODO 3–4 уходят в домашнее задание.

---
## 0. Установка зависимостей

In [1]:
!pip install -q "openai>=1.40,<2" "pydantic>=2.7,<3" "pandas>=2.0" "openpyxl>=3.1"

---
## 1. Учебные данные

Файл `dirty_orders.xlsx` — три листа с заявками, справочник станций `stations.json` (120 станций:
код, название, дорога) и файл с контрольными суммами для `check()`.
Контрольные суммы — это хэши: по ним проверка узнаёт, совпало ли ваше решение, но не раскрывает его.

In [3]:
# @title Учебные данные
import calendar
import hashlib
import json
import os
import re
import string
from pathlib import Path

import openpyxl

# Каталог notebooks/data в публичном репозитории курса.
COURSE_DATA_URL = "https://raw.githubusercontent.com/MaximSantalov/ai_miit/main/notebooks/data"
DATA_FILES = ("dirty_orders.xlsx", "stations.json", "orders_sheet_checks.json")


def _find_local_data_dir():
    candidates = [os.environ.get("COURSE_DATA_DIR"), "notebooks/data", "data", "../data", "."]
    for c in candidates:
        if not c:
            continue
        p = Path(c)
        if all((p / f).exists() for f in DATA_FILES):
            return p
    return None


def ensure_data() -> Path:
    """Возвращает каталог с учебными данными, при необходимости скачивая их."""
    local = _find_local_data_dir()
    if local is not None:
        return local

    import urllib.request

    target = Path("data")
    target.mkdir(exist_ok=True)
    for f in DATA_FILES:
        try:
            urllib.request.urlretrieve(f"{COURSE_DATA_URL}/{f}", target / f)
        except Exception as e:  # noqa: BLE001
            raise RuntimeError(
                f"Не удалось скачать {f} из репозитория курса ({type(e).__name__}: {e}).\n"
                f"Что сделать: проверьте COURSE_DATA_URL в этой ячейке или положите файлы "
                f"{list(DATA_FILES)} рядом с ноутбуком в каталог data/ вручную."
            ) from e
    return target


DATA_DIR = ensure_data().resolve()
WB = openpyxl.load_workbook(DATA_DIR / "dirty_orders.xlsx")
SHEETS = WB.sheetnames
STATIONS = json.loads((DATA_DIR / "stations.json").read_text(encoding="utf-8"))
STATION_NAMES = [s["station_name"] for s in STATIONS]
PUBLIC = json.loads((DATA_DIR / "orders_sheet_checks.json").read_text(encoding="utf-8"))
TODAY = "2026-10-15"


def load_sheet(sheet: str) -> list[tuple[int, list[str]]]:
    """Все строки листа: (номер строки Excel с 1, список ячеек-строк; пустая ячейка — "")."""
    return [(i, ["" if c is None else str(c) for c in row])
            for i, row in enumerate(WB[sheet].iter_rows(values_only=True), 1)]


def col_index(letter: str) -> int:
    """Буква колонки Excel -> индекс в списке ячеек строки: A -> 0, B -> 1, ..."""
    return string.ascii_uppercase.index(letter.upper())


def canon_hash(value) -> str:
    """SHA-256 от канонического JSON — так устроены контрольные суммы в check()."""
    text = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


print("данные лежат в:", DATA_DIR)
print("листы:", SHEETS, "| станций в справочнике:", len(STATIONS))

данные лежат в: /content/data
листы: ['Реестр_заявок_1', 'Реестр_заявок_2', 'Выгрузка_raw'] | станций в справочнике: 120


---
## 2. Подключение к курсовому прокси

Как на практиках 1–3: библиотека `openai`, курсовой прокси, модель `course-fast`.

In [4]:
# @title Подключение к прокси
import getpass

from openai import OpenAI

# ── Куда ходим и какой моделью ──────────────────────────────────────────
BASE_URL = "https://llm.kamani.tech/v1"
MODEL = "course-fast"

# Если course-fast не умеет инструменты — следующая ячейка скажет, переключайтесь:
# MODEL = "course-smart"


def connect(reset: bool = False) -> OpenAI:
    """Создаёт клиента. connect(reset=True) — ввести ключ заново."""
    global client
    if reset:
        os.environ.pop("COURSE_LLM_API_KEY", None)
    if not os.environ.get("COURSE_LLM_BASE_URL"):
        os.environ["COURSE_LLM_BASE_URL"] = (
            BASE_URL or input("base_url (например https://.../v1): ").strip()
        )
    if not os.environ.get("COURSE_LLM_API_KEY"):
        os.environ["COURSE_LLM_API_KEY"] = getpass.getpass("Ваш ключ (ввод не виден): ").strip()
    client = OpenAI(
        base_url=os.environ["COURSE_LLM_BASE_URL"],
        api_key=os.environ["COURSE_LLM_API_KEY"],
    )
    print("Клиент создан. base_url =", os.environ["COURSE_LLM_BASE_URL"], "| модель:", MODEL)
    return client


client = connect()

Ваш ключ (ввод не виден): ··········
Клиент создан. base_url = https://llm.kamani.tech/v1 | модель: course-fast


In [5]:
# @title Проверка: умеет ли модель вызывать инструменты
PROBE_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_wagons_on_station",
        "description": "Возвращает количество вагонов на станции прямо сейчас.",
        "parameters": {
            "type": "object",
            "properties": {"station": {"type": "string", "description": "Название станции"}},
            "required": ["station"],
        },
    },
}]


def check_tools(model: str = MODEL) -> bool:
    """Преполётная проверка: умеет ли модель возвращать запрос на вызов инструмента."""
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user",
                       "content": "Сколько вагонов сейчас на станции ДОЛИНО? "
                                  "Ответь, воспользовавшись инструментом."}],
            tools=PROBE_TOOL, tool_choice="auto", max_tokens=200,
        )
    except Exception as e:  # noqa: BLE001
        print(f"❌ Пробный запрос не прошёл: {type(e).__name__}: {e}")
        print("   Что делать: проверьте ключ (connect(reset=True)) или поменяйте MODEL.")
        return False
    if not r.choices[0].message.tool_calls:
        print(f"❌ Модель '{model}' ответила текстом вместо вызова. Поменяйте MODEL на "
              "'course-smart' в ячейке выше и повторите.")
        return False
    print(f"✅ Модель '{model}' умеет вызывать инструменты.")
    return True


check_tools()

✅ Модель 'course-fast' умеет вызывать инструменты.


True

---
## 3. Как ноутбук сообщает о невыполненных заданиях

Как на практиках 2–3: пока ячейка с заданием не дописана, «Проверка» не падает с трассировкой, а
говорит, что именно не сделано.

In [6]:
# @title Сообщения о невыполненных заданиях
class TodoNotDone(NotImplementedError):
    """Помеченная ячейка # TODO ещё не дописана."""


def need(value, number: int, what: str):
    """Проверяет результат вашей функции. Готовый код, менять не нужно."""
    if value is None or (isinstance(value, (str, list, dict)) and len(value) == 0):
        raise TodoNotDone(f"TODO {number} не выполнен: {what}")
    return value


def explain_todo(e: TodoNotDone) -> None:
    print("⛔", e)
    print("   Что делать: допишите ячейку с заданием выше, выполните её (Shift+Enter)")
    print("   и запустите эту ячейку заново.")

---
## 4. Что внутри файла

Первые строки каждого листа — как их видит человек, открывший Excel. Номера слева — номера строк
в Excel, буквы сверху — колонки.

In [7]:
# @title Первые строки каждого листа
def show_sheet(sheet: str, start: int = 1, n: int = 10) -> None:
    rows = [r for r in load_sheet(sheet) if start <= r[0] < start + n]
    print(f"── {sheet} ──   (строки {start}–{start + n - 1} из {len(load_sheet(sheet))})")
    for row_no, cells in rows:
        shown = " | ".join((c[:22] or "·") for c in cells)
        print(f"{row_no:>3}  {shown}")
    print()


for _sheet in SHEETS:
    show_sheet(_sheet)

── Реестр_заявок_1 ──   (строки 1–10 из 55)
  1  ООО «ТрансЛогистика» — | · | · | · | · | · | · | ·
  2  Дата формирования отчё | · | · | · | · | · | · | ·
  3  Утвердил: нач. отдела  | · | · | · | · | · | · | ·
  4  · | · | · | · | · | · | · | ·
  5  · | · | · | · | Маршрут | · | · | ·
  6  № | Клиент | Груз | Род вагона | Ст. отправления | Ст. назначения | Кол-во вагонов | Дата подачи
  7  1 | ЗернoТрейд Юг | зерно | хоппер-минераловоз | Инская | Песчаная-Грузовая | 29 | 03.08.2025
  8  2 | Кузбасс-Экспорт | уголь каменный | полувагон | Мариинск | Туапсе-Сортировочная | 18 шт | 2025-04-10
  9  3 | АгроЗернопродукт | зерно | хоппер-минераловоз | Бор-Сортировочная | Ельник-Южная | 18 ваг. | середина октября 2025
 10  4 |   РудаИнвест   | руда железная | полувагон | Кемерово-Сортировочное | Починок-Грузовая | · | середина декабря

── Реестр_заявок_2 ──   (строки 1–10 из 46)
  1  Реестр заявок за март  | · | · | · | · | · | ·
  2  · | · | · | · | · | · | ·
  3  Заказчик | Груз | Тип ваго

In [8]:
# @title Проверки check()
SQL_WORDS = re.compile(r"\b(select|insert|update|delete|drop|union|where|from)\b|;|--", re.I)
HEX = re.compile(r"^[0-9a-f]{64}$")


def _check_1():
    if not LAYOUTS:
        return False, "нет раскладок: не выполнена ячейка «Проверка TODO 1»"
    bad = []
    for sheet in SHEETS:
        got = LAYOUTS.get(sheet)
        if got is None:
            bad.append(f"{sheet}: субагент не вернул раскладку")
        elif canon_hash(got) != PUBLIC["layout_hash"][sheet]:
            bad.append(f"{sheet}: раскладка не совпала с эталоном")
    if bad:
        return False, "; ".join(bad)
    return True, "три раскладки листов совпали с эталоном (шапка в строке 6 / 3 / нет)"


def _normalized():
    if len(LAYOUTS) < len(SHEETS):
        raise TodoNotDone("TODO 1 не выполнен: нет раскладок листов — сначала «Проверка TODO 1» "
                          "(раскладку можно вписать и руками: LAYOUTS[лист] = {...})")
    return {s: normalize_sheet(s) for s in SHEETS}


def _check_2():
    norm = _normalized()
    total_rows = sum(len(v) for v in norm.values())
    per_sheet = {s: len(v) for s, v in norm.items()}
    if per_sheet != PUBLIC["counts"]:
        hints = []
        for sheet in SHEETS:
            got, want = per_sheet[sheet], PUBLIC["counts"][sheet]
            if got < want:
                hints.append(f"{sheet}: {got} вместо {want} — выброшено лишнее; «ИТОГО» искали подстрокой?")
            elif got > want:
                hints.append(f"{sheet}: {got} вместо {want} — остались пустые строки, «ИТОГО» или дубли")
        return False, "; ".join(hints)
    if total_rows != PUBLIC["total"]:
        return False, f"уникальных заявок {total_rows}, а должно быть {PUBLIC['total']}"
    if not any("Магнитогорск-Грузовой" in (o["to_station"], o["from_station"])
               for v in norm.values() for o in v):
        return False, "пропала заявка со станцией «Магнитогорск-Грузовой» — «ИТОГО» искали подстрокой?"
    return True, "120 уникальных заявок (45 + 40 + 35), «ИТОГО» нет, «Магнитогорск-Грузовой» на месте"


def _check_3():
    norm = _normalized()
    bad = []
    for sheet, orders in norm.items():
        for o in orders:
            for key in ("date_from", "date_to"):
                v = o[key]
                if v is not None and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(v)):
                    return False, f"{sheet}, строка {o['row']}: дата {v!r} не в формате ГГГГ-ММ-ДД"
            if o["wagons"] is not None and (not isinstance(o["wagons"], int)
                                            or isinstance(o["wagons"], bool)):
                return False, f"{sheet}, строка {o['row']}: количество {o['wagons']!r} — не int и не None"
            ref = PUBLIC["row_hash"].get(f"{sheet}:{o['row']}")
            mine = canon_hash([o["client"], o["cargo"], o["wagon_type"], o["wagons"],
                               o["date_from"], o["date_to"]])
            if ref != mine:
                bad.append(f"{sheet}:{o['row']}")
    if bad:
        return False, (f"{len(bad)} строк не совпали с эталоном (клиент, груз, род вагона, "
                       f"количество или даты), например {', '.join(bad[:4])}")
    return True, "даты ISO, количество int или None, год для дат без года выбран по правилу"


def _check_4():
    if not FINAL_ORDERS:
        return False, "не выполнена ячейка «Сборка базы»"
    if UNKNOWN_STATIONS:
        return False, f"станции не найдены в справочнике: {sorted(set(UNKNOWN_STATIONS))}"
    typos = {canon_hash(t): canon_hash(f) for t, f in FIXED_STATIONS}
    if typos != PUBLIC["typos"]:
        return False, f"исправлено {len(FIXED_STATIONS)} станций, а опечаток в файле 4 — или исправлены неверно"
    bad = [k for k, v in PUBLIC["station_hash"].items()
           if canon_hash([FINAL_BY_KEY[k]["from_station"], FINAL_BY_KEY[k]["to_station"]]) != v]
    if bad:
        return False, f"станции не совпали с эталоном в строках: {', '.join(bad[:4])}"
    return True, "4 опечатки исправлены, неизвестных 0, верные станции не тронуты"


def _check_5():
    for sheet in SHEETS:
        _, _, tools, _ = make_subagent(sheet)
        names = {t["function"]["name"] for t in (tools or [])}
        if "spawn_agent" in names:
            return False, "у субагента есть spawn_agent — рекурсия запрещена (MAX_NESTED_SPAWN = 0)"
    fresh = {"agents": 0, "completion_tokens": 0, "depth": 0}
    if check_limits(dict(fresh), SHEETS[0]) is not None:
        return False, "на свежем состоянии check_limits вернул отказ — лимиты сработали раньше времени"
    cases = {
        "агентов уже MAX_AGENTS": {**fresh, "agents": MAX_AGENTS},
        "токены исчерпаны": {**fresh, "completion_tokens": MAX_TOTAL_COMPLETION_TOKENS},
        "вызов из субагента (depth 1)": {**fresh, "depth": 1},
    }
    for what, state in cases.items():
        try:
            reply = check_limits(dict(state), SHEETS[0])
        except Exception as e:  # noqa: BLE001
            return False, f"{what}: исключение {type(e).__name__} вместо текста ошибки"
        if not isinstance(reply, str) or not reply.strip():
            return False, f"{what}: нужен непустой текст ошибки, а вернулось {reply!r}"
    if not LIMIT_REPLY or not isinstance(LIMIT_REPLY, str):
        return False, "не выполнена ячейка «Проверка TODO 4»"
    return True, "четвёртый вызов получил текст ошибки, у субагентов нет spawn_agent"


def _check_6():
    if not CONTROL_RUN:
        return False, "не выполнена ячейка «Контрольный вопрос»"
    calls = [c for c in CONTROL_RUN["calls"] if c["name"] == "query_orders"]
    if not calls:
        return False, "агент не позвал query_orders"
    for c in calls:
        if any(SQL_WORDS.search(str(v)) for v in c["arguments"].values()):
            return False, f"в аргументах query_orders есть SQL: {c['arguments']}"
    if not has_number(CONTROL_RUN["answer"], PUBLIC["control"]["wagons"]):
        return False, f"в ответе нет числа {PUBLIC['control']['wagons']}: {CONTROL_RUN['answer'][:120]!r}"
    return True, f"агент ответил через query_orders, число {PUBLIC['control']['wagons']} совпало, SQL в аргументах нет"


CHECK_LIST = [("TODO 1 · раскладки листов", _check_1), ("TODO 2 · 120 уникальных заявок", _check_2),
              ("TODO 2 · даты и количество", _check_3), ("TODO 3 · станции по справочнику", _check_4),
              ("TODO 4 · ограничители", _check_5), ("Контрольный вопрос через query_orders", _check_6)]


def check(*numbers: int) -> int:
    """check() — все шесть проверок; check(1, 2) — только указанные."""
    chosen = numbers or tuple(range(1, len(CHECK_LIST) + 1))
    passed = 0
    for i in chosen:
        title, fn = CHECK_LIST[i - 1]
        try:
            ok, msg = fn()
        except TodoNotDone as e:
            ok, msg = False, str(e)
        except Exception as e:  # noqa: BLE001
            ok, msg = False, f"{type(e).__name__}: {e}"
        passed += ok
        print(f"{'✅' if ok else '❌'} {i}. {title}: {msg}")
    print(f"Итого: {passed} из {len(chosen)}")
    return passed


# Результаты ячеек до их запуска пустые. Ready-ячейки только дополняют эти объекты.
LAYOUTS: dict = {}
AGENTS_LOG: list = []
SPAWN_STATE = {"agents": 0, "completion_tokens": 0, "depth": 0}
ORCH_RUN: dict = {}
FINAL_ORDERS: list = []
FINAL_BY_KEY: dict = {}
FIXED_STATIONS: list = []
UNKNOWN_STATIONS: list = []
LIMIT_REPLY = ""
CONTROL_RUN: dict = {}
SINGLE_RUN: dict = {}
print("Проверки готовы.")

Проверки готовы.


---
## 5. Цикл агента — готовый

Это цикл из практики 2: запрос → просьба вызвать инструмент → вызов → результат в переписку →
снова запрос. Сегодня он работает на двух уровнях: оркестратор и субагент — это один и тот же
`run_agent` с разными инструкцией, инструментами и моделью. Параметр `answer_format` просит у модели
ответ по JSON-схеме — так субагент возвращает раскладку листа.

In [9]:
# @title Цикл агента
import time

def assistant_message(msg) -> dict:
    out = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        out["tool_calls"] = [
            {"id": c.id, "type": "function",
             "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls
        ]
    return out


def parse_arguments(tool_call) -> dict:
    try:
        return json.loads(tool_call.function.arguments or "{}")
    except json.JSONDecodeError:
        return {}


def _short(text: str, n: int = 150) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[: n - 1] + "…"


def _cost(header) -> float | None:
    """Цену запроса в долларах сообщает курсовой прокси в заголовке ответа."""
    try:
        return float(header) if header not in (None, "") else None
    except ValueError:
        return None


def total(result_or_results, key: str):
    """Сумма поля usage по одному прогону или по списку прогонов."""
    runs = result_or_results if isinstance(result_or_results, list) else [result_or_results]
    values = [u.get(key) for r in runs for u in r.get("usage", [])]
    if key == "cost" and any(v is None for v in values):
        return None
    return sum(v or 0 for v in values)


def per_thousand(value, unit: str) -> str:
    """Сколько это на 1000 таких запросов — чтобы разница ощущалась."""
    if value is None:
        return "—"
    if unit == "$":
        return f"${value * 1000:,.2f}".replace(",", " ")
    return f"{value * 1000 / 1e6:,.1f} млн".replace(",", " ")


def _usage(r) -> dict:
    u = r.usage
    det = getattr(u, "completion_tokens_details", None)
    reasoning = (getattr(det, "reasoning_tokens", 0) or 0) if det else 0
    return {"prompt": u.prompt_tokens, "completion": u.completion_tokens, "reasoning": reasoning}


def run_agent(question: str, system: str | None = None, max_steps: int = 8,
              model: str | None = None, verbose: bool = True, *,
              tools: list | None = None, call=None, history: list | None = None,
              before_call=None, reasoning: str | None = None,
              answer_format: dict | None = None) -> dict:
    """Цикл агента практики 2. Готовый код."""
    model = model or MODEL
    if history:
        messages = list(history)
    else:
        messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})
    calls, usage = [], []
    extra = {"reasoning_effort": reasoning} if reasoning else {}
    if answer_format:
        extra["response_format"] = answer_format

    for step in range(1, max_steps + 1):
        if before_call is not None:
            messages = before_call(messages)
        try:
            started = time.perf_counter()
            raw = client.chat.completions.with_raw_response.create(
                model=model, messages=messages, temperature=0,
                **({"tools": tools, "tool_choice": "auto"} if tools else {}), **extra)
            r = raw.parse()
            seconds = time.perf_counter() - started
        except Exception as e:  # noqa: BLE001
            if reasoning and "reasoning" in str(e).lower():
                raise RuntimeError(f"Модель {model} не принимает reasoning_effort: {e}") from e
            raise
        msg = r.choices[0].message
        usage.append({"step": step, **_usage(r), "seconds": seconds,
                      "cost": _cost(raw.headers.get("x-litellm-response-cost"))})
        if verbose:
            u = usage[-1]
            extra_info = f" · рассуждение {u['reasoning']} ток." if u["reasoning"] else ""
            print(f"  шаг {step} · в модель ушло {u['prompt']} ток. контекста{extra_info}")
            if msg.tool_calls and msg.content:
                print(f"          модель: {_short(msg.content)}")

        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content or ""})
            return {"answer": (msg.content or "").strip(), "steps": step, "calls": calls,
                    "stopped_by": "text", "usage": usage, "messages": messages}

        messages.append(assistant_message(msg))
        for c in msg.tool_calls:
            args = parse_arguments(c)
            result = call(c.function.name, args)
            calls.append({"step": step, "name": c.function.name, "arguments": args,
                          "result": result})
            if verbose:
                print(f"          {c.function.name}({json.dumps(args, ensure_ascii=False)})")
                print(f"            -> {_short(result, 120)}")
            messages.append({"role": "tool", "tool_call_id": c.id, "content": result})

    return {"answer": "", "steps": max_steps, "calls": calls, "stopped_by": "max_steps",
            "usage": usage, "messages": messages}


def reasoning_tokens(result: dict) -> int:
    return sum(u["reasoning"] for u in result.get("usage", []))


def has_number(text: str, n: int) -> bool:
    """Есть ли в тексте число n — с пробелами между разрядами или без."""
    flat = "".join(ch for ch in str(text) if not ch.isspace() and ch not in "  ")
    return str(n) in flat


print("Цикл готов.")

Цикл готов.


---
## 6. Инструмент субагента и схема раскладки

`read_sheet(sheet, start, n)` — единственный инструмент субагента: возвращает `n` строк листа
начиная со строки `start`. Лист целиком в модель не уходит.

`SheetLayout` — то, что должен вернуть субагент: в какой строке заголовки (`header_row`, а если их
нет — `None`), с какой строки начинаются данные (`first_data_row`) и в какой колонке какое поле.
Поля одинаковы для всех листов: `client`, `cargo`, `wagon_type`, `from_station`, `to_station`,
`wagons`, `date`.

In [10]:
# @title read_sheet, SheetLayout, схема ответа
from typing import Optional

from pydantic import BaseModel, ConfigDict, Field

# Защита от запуска ячеек вне порядка: SHEETS обычно создаётся в ячейке «Учебные данные».
if "SHEETS" not in globals():
    if "WB" not in globals():
        raise RuntimeError("Сначала выполните ячейку «Учебные данные»: она загружает Excel и список листов.")
    SHEETS = list(WB.sheetnames)

FIELDS = ["client", "cargo", "wagon_type", "from_station", "to_station", "wagons", "date"]


class Columns(BaseModel):
    model_config = ConfigDict(extra="forbid")
    client: str
    cargo: str
    wagon_type: str
    from_station: str
    to_station: str
    wagons: str
    date: str


class SheetLayout(BaseModel):
    """Раскладка листа: номер строки заголовков, первая строка данных, буква колонки на поле."""
    model_config = ConfigDict(extra="forbid")
    header_row: Optional[int]
    first_data_row: int
    columns: Columns


LAYOUT_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "sheet_layout", "strict": True,
    "schema": {
        "type": "object", "additionalProperties": False,
        "required": ["header_row", "first_data_row", "columns"],
        "properties": {
            "header_row": {"type": ["integer", "null"]},
            "first_data_row": {"type": "integer"},
            "columns": {"type": "object", "additionalProperties": False, "required": FIELDS,
                        "properties": {f: {"type": "string"} for f in FIELDS}},
        }}}}


def read_sheet(sheet: str, start: int = 1, n: int = 10) -> str:
    """Строки листа start … start+n-1: номер строки Excel и ячейки; сверху — буквы колонок."""
    if sheet not in SHEETS:
        return f"Ошибка: нет листа «{sheet}». Есть: {SHEETS}."
    start, n = max(int(start), 1), min(max(int(n), 1), 30)
    rows = [r for r in load_sheet(sheet) if start <= r[0] < start + n]
    width = max((len(v) for _, v in load_sheet(sheet)), default=0)
    lines = ["строка\t" + "\t".join(string.ascii_uppercase[:width])]
    lines += [f"{no}\t" + "\t".join(cells) for no, cells in rows]
    return "\n".join(lines)


READ_SHEET_TOOL = {"type": "function", "function": {
    "name": "read_sheet",
    "description": ("Возвращает n строк листа Excel, начиная со строки start. В первой строке — буквы колонок, "
                    "дальше номер строки Excel и значения ячеек через табуляцию. Пустая ячейка — пустое место."),
    "parameters": {"type": "object", "required": ["sheet"], "properties": {
        "sheet": {"type": "string", "enum": SHEETS, "description": "название листа"},
        "start": {"type": "integer", "description": "первая строка, по умолчанию 1"},
        "n": {"type": "integer", "description": "сколько строк, не больше 30, по умолчанию 10"}}}}}


def sheet_call(name: str, args: dict) -> str:
    """Выполняет вызов инструмента субагента. Других инструментов у него нет."""
    if name != "read_sheet":
        return f"Ошибка: у субагента нет инструмента «{name}». Есть только read_sheet."
    try:
        return read_sheet(**args)
    except TypeError as e:
        return f"Ошибка в аргументах read_sheet: {e}"


print(read_sheet(SHEETS[2], 1, 3))

строка	A	B	C	D	E	F	G
1	МеталлКокс Логистика	кокс	полувагон	Яр-Сортировочная	Кемерово-Сортировочное		01.10.2025
2	КонтейнерЛайн	контейнеры	фитинговая платформа	Гранитная-Западная	Орехово-Зуево	44	2025-08-10
3	УгольТрансСервис	уголь каменный	полувагон	Новокузнецк-Восточный	Новороссийск-Экспорт	48 шт	середина октября 2025


---
# TODO 1 · Субагент — это четыре поля

## Зачем

Оркестратор не читает лист сам. Он зовёт `spawn_agent(sheet)`, а тот собирает **субагента из
четырёх полей**: инструкция, контекст, инструменты, модель. Субагент отвечает на один вопрос про
один лист: *где шапка и в какой колонке что лежит*. Это перевод из одной формы в другую, и именно в
нём модель сильна. Числа и даты она считать не будет — этим займётся код.

## Что написать: только инструкцию

Три поля из четырёх уже заполнены в ячейке ниже: `context` (первые 10 строк листа), `tools`
(`[READ_SHEET_TOOL]`: субагент может дочитать лист; `spawn_agent` ему не выдаётся) и `model`
(`MODEL`). Вам остаётся одно: **написать текст инструкции** для субагента — 5–10 строк обычным языком.

В инструкции должно быть:

1. **Что сделать:** определить раскладку листа с заявками на подачу вагонов.
2. **`header_row`** — номер строки с заголовками столбцов (Клиент, Груз, …). Заголовок над несколькими
   колонками («Маршрут») строкой заголовков не считается. Если заголовков нет — `None` (`null`).
3. **`first_data_row`** — номер первой строки с данными заявки (не шапка документа, не заголовки,
   не пустая строка).
4. **`columns`** — буква колонки для каждого из семи полей: `client`, `cargo`, `wagon_type`,
   `from_station`, `to_station`, `wagons`, `date`. Если заголовков нет — определять поле по значениям:
   названия станций, числа, даты.
5. **Критерий успеха:** все семь полей указаны, буквы верны; ничего не придумывать, а если десяти строк
   мало — дочитать лист инструментом `read_sheet`.

Форму ответа (JSON) модель получает от схемы `SheetLayout` — про JSON писать не нужно.

In [11]:
# ═══════════════════════════ TODO 1 из 4 ═══════════════════════════
# Напишите текст инструкции для субагента (переменная instruction, 5–10 строк).
# context, tools и model уже заполнены. Выполните ячейку, затем «Проверка TODO 1» —
# она запустит оркестратора и трёх субагентов.
# ═══════════════════════════════════════════════════════════════════

def make_subagent(sheet: str):
    instruction = f"""\
Ты разбираешь один лист Excel «{sheet}» с заявками на подачу вагонов. Ниже — первые 10 строк листа.
Определи раскладку листа: где кончается шапка и в какой колонке какое поле лежит.
header_row — номер строки Excel с заголовками столбцов (Клиент, Груз, Род вагона и т. д.). Заголовок над несколькими колонками («Маршрут») строкой заголовков не считается. Если заголовков на листе нет вовсе — верни null.
first_data_row — номер первой строки с данными заявки. Это не шапка документа (название, дата формирования, «Утвердил»), не строка заголовков и не пустая строка.
columns — буква колонки (A, B, C…) для каждого из семи полей: client, cargo, wagon_type, from_station, to_station, wagons, date. Колонка «№» — это номер строки, она ни к одному полю не относится. Если заголовков нет, определяй поле по значениям: название компании — client, груз — cargo, тип вагона — wagon_type, две станции подряд — from_station и to_station, число (иногда с «шт» или «ваг.») — wagons, дата — date.
Успех: указаны все семь полей, буквы верны, номера строк — это номера строк Excel из первой колонки. Ничего не придумывай: если десяти строк не хватает, дочитай лист инструментом read_sheet."""
    context = read_sheet(sheet, 1, 10)   # первые 10 строк листа, а не весь лист
    tools = [READ_SHEET_TOOL]            # субагент может дочитать лист; spawn_agent ему не выдан
    model = MODEL
    return instruction, context, tools, model

---
## 7. Оркестратор — готовый

`spawn_agent(sheet)` — инструмент оркестратора. Он вызывает ваш `make_subagent`, запускает субагента
циклом `run_agent` и кладёт его ответ в словарь `LAYOUTS`. Оркестратор — тоже `run_agent`, но его
единственный инструмент — `spawn_agent`: он **делегирует и заканчивает**, а в данных сам ничего не
делает.

Перед запуском каждого субагента `spawn_agent` спрашивает `check_limits` (TODO 4). Пока вы его не
написали, ограничителей нет.

In [12]:
# @title spawn_agent и оркестратор
MAX_AGENTS = 3
MAX_NESTED_SPAWN = 0
MAX_TOTAL_COMPLETION_TOKENS = 2048


def check_limits(state: dict, sheet: str):
    """Заглушка до TODO 4: ограничителей нет."""
    return None


def spawn_agent(sheet: str) -> str:
    """Инструмент оркестратора: собирает субагента листа, запускает его, возвращает раскладку."""
    if sheet not in SHEETS:
        return f"Ошибка: нет листа «{sheet}». Есть: {SHEETS}."
    refusal = check_limits(SPAWN_STATE, sheet)
    if refusal:
        return refusal
    instruction, context, tools, model = make_subagent(sheet)
    need(instruction, 1, "make_subagent вернул пустую instruction")
    need(context, 1, "make_subagent вернул пустой context")
    need(tools, 1, "make_subagent вернул пустой tools — нужен хотя бы [READ_SHEET_TOOL]")
    need(model, 1, "make_subagent вернул пустую model")
    SPAWN_STATE["agents"] += 1
    run = run_agent(context, system=instruction, tools=tools, call=sheet_call, model=model,
                    answer_format=LAYOUT_FORMAT, max_steps=4, verbose=False)
    SPAWN_STATE["completion_tokens"] += total(run, "completion")
    AGENTS_LOG.append({"sheet": sheet, "usage": run["usage"], "steps": run["steps"]})
    try:
        layout = SheetLayout.model_validate_json(run["answer"])
    except Exception as e:  # noqa: BLE001
        return f"Ошибка: субагент листа «{sheet}» вернул не раскладку ({type(e).__name__})."
    LAYOUTS[sheet] = layout.model_dump()
    return json.dumps(layout.model_dump(), ensure_ascii=False)


SPAWN_TOOL = [{"type": "function", "function": {
    "name": "spawn_agent",
    "description": ("Запускает субагента, который разбирает ОДИН лист Excel и возвращает его раскладку: "
                    "строку заголовков, первую строку данных и колонки полей. Один вызов — один лист."),
    "parameters": {"type": "object", "required": ["sheet"],
                   "properties": {"sheet": {"type": "string", "enum": SHEETS,
                                            "description": "название листа"}}}}}]

ORCH_SYSTEM = f"""\
Ты — оркестратор. В файле три листа: {', '.join(SHEETS)}. Для каждого листа вызови spawn_agent
ровно один раз; сам листы не читай и раскладки не придумывай. Если инструмент вернул ошибку —
перескажи её одной строкой. Когда ответы получены, напиши коротко: что получилось по каждому листу."""


def orchestrate(verbose: bool = True) -> dict:
    """Оркестратор раздаёт листы субагентам. Состояние лимитов начинается с нуля."""
    SPAWN_STATE.update(agents=0, completion_tokens=0, depth=0)
    LAYOUTS.clear()
    AGENTS_LOG.clear()

    def call(name, args):
        if name != "spawn_agent":
            return f"Ошибка: у оркестратора нет инструмента «{name}»."
        return spawn_agent(str(args.get("sheet", "")))

    run = run_agent("Разбери все листы файла.", system=ORCH_SYSTEM, tools=SPAWN_TOOL, call=call,
                    max_steps=8, verbose=verbose)
    ORCH_RUN.clear()
    ORCH_RUN.update(run)
    return run


print("Оркестратор готов.")

Оркестратор готов.


## Проверка TODO 1

Оркестратор запускает трёх субагентов — по одному на лист. Каждый видит десять строк и возвращает
раскладку. Проверка сверяет раскладки с эталоном: на листе 1 шапка в строке 6, на листе 2 — в
строке 3, на листе 3 заголовков нет.

In [13]:
# Проверка TODO 1
try:
    need(make_subagent(SHEETS[0])[0], 1, "make_subagent вернул пустую instruction — допишите функцию")
    print("─── оркестратор ───")
    orchestrate()
    print("\nответ оркестратора:", ORCH_RUN["answer"])
    for sheet, layout in LAYOUTS.items():
        print(f"  {sheet}: шапка {layout['header_row']}, данные с {layout['first_data_row']}")
    print()
    check(1)
except TodoNotDone as e:
    explain_todo(e)

─── оркестратор ───
  шаг 1 · в модель ушло 471 ток. контекста
          модель: I'll spawn one agent per sheet, all in parallel.
          spawn_agent({"sheet": "Реестр_заявок_1"})
            -> {"header_row": 6, "first_data_row": 7, "columns": {"client": "B", "cargo": "C", "wagon_type": "D", "from_station": "E",…
          spawn_agent({"sheet": "Реестр_заявок_2"})
            -> {"header_row": 3, "first_data_row": 4, "columns": {"client": "A", "cargo": "B", "wagon_type": "C", "from_station": "D",…
          spawn_agent({"sheet": "Выгрузка_raw"})
            -> {"header_row": null, "first_data_row": 1, "columns": {"client": "A", "cargo": "B", "wagon_type": "C", "from_station": "…
  шаг 2 · в модель ушло 831 ток. контекста

ответ оркестратора: Готово, все три листа разобраны (по одному вызову агента на лист, ошибок не было).

- **Реестр_заявок_1** — заголовки в строке 6, данные с 7-й. Колонки: клиент B, груз C, тип вагона D, станция отправления E, станция назначения F, вагоны G, дата 

In [14]:
# Контрольная точка 1
import pandas as pd

rows = []
for sheet in SHEETS:
    lay = LAYOUTS.get(sheet)
    rows.append({"лист": sheet,
                 "шапка": "—" if not lay else lay["header_row"],
                 "данные с": "—" if not lay else lay["first_data_row"],
                 **({} if not lay else {f: lay["columns"][f] for f in FIELDS})})
pd.DataFrame(rows).set_index("лист")

,шапка,данные с,client,cargo,wagon_type,from_station,to_station,wagons,date
лист,,,,,,,,,
Реестр_заявок_1,6.0,7,B,C,D,E,F,G,H
Реестр_заявок_2,3.0,4,A,B,C,D,E,F,G
Выгрузка_raw,NaN,1,A,B,C,D,E,F,G


---
# TODO 2 · Числа считает код

## Зачем

Раскладка листа — это работа модели. Дальше начинаются числа и форматы: даты в четырёх видах,
количество в четырёх стилях, «ИТОГО» и дубли. Здесь ошибки модели недопустимы: один и тот же файл
должен давать одни и те же 120 заявок, поэтому это пишет **код**.

## Что в файле

Откройте любой лист глазами (ячейка «Что внутри файла») и найдите:

- **пустые строки** между строками данных и **строки «ИТОГО»** (ячейка равна «ИТОГО» целиком —
  а не содержит эти буквы: есть станция «Магни**ИТОГО**рск-Грузовой»);
- **количество:** `29`, `18 шт`, `18 ваг.`, пустая ячейка;
- **дата подачи:** `03.08.2025`, `2025-04-10`, `середина октября 2025`, `начало марта` (без года);
- лишние **пробелы** вокруг клиента и **регистр** груза (`УГОЛЬ КАМЕННЫЙ`);
- **точные дубли** строк.

## Что написать: `normalize(rows, layout)`

`rows` — все строки листа `[(номер_строки, [ячейки]), …]`, `layout` — раскладка из TODO 1
(словарь: `header_row`, `first_data_row`, `columns` — буква колонки на поле). Верните список словарей:

```python
{"row": номер строки, "client": …, "cargo": …, "wagon_type": …,
 "from_station": …, "to_station": …, "wagons": int | None, "date_from": …, "date_to": …}
```

Правила:

1. Берите строки с `first_data_row` и дальше; **пустые** и «ИТОГО» пропускайте.
2. `client` — без внешних пробелов и лишних пробелов внутри; `cargo`, `wagon_type` — без внешних
   пробелов, строчными буквами; станции — без внешних пробелов (опечатки — в TODO 3).
3. `wagons` — `int`; пустая ячейка — `None`: значение не восстанавливаем из соседних строк.
4. Дата — **два поля**, начало и конец периода, ISO `ГГГГ-ММ-ДД`. Точная дата: `date_from == date_to`.
   «начало / середина / конец месяца» — период по таблице `PERIODS`. Год берите из строки; если его
   нет — из `sheet_year(rows, layout)`. Если год неизвестен совсем — `None` в обоих полях.
5. **Дубли** (все поля, кроме номера строки, совпали) — оставьте первую, остальные выбросьте
   **после** приведения форм: `"15 шт"` и `"15"` — одно и то же.

Готово: `MONTHS`, `PERIODS`, `sheet_year`, `col_index`.

**Что дано, что дописать:** каркас `normalize` и `parse_date` готов. Допишите **4 места** с пометкой `ВАШ КОД (1…4)` — всего около 8 строк. «Диагностика TODO 2» проверяет `parse_wagons` и `parse_date` на примерах без модели и Excel и показывает, какие строки отброшены.

In [15]:
# @title Помощники для TODO 2: месяцы, периоды, год листа
MONTHS = {"января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6, "июля": 7,
          "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12}
# Текстовое приближение даты -> (первый день, последний день); None — последний день месяца.
PERIODS = {"начало": (1, 10), "середина": (11, 20), "конец": (21, None)}


def sheet_year(rows, layout) -> tuple[int | None, str]:
    """Год для дат без года и откуда он взят.
    1) явный год в шапке листа (дата формирования, название) — берём последний;
    2) иначе единственный год, который встречается в полных датах самого листа;
    3) иначе год неизвестен: (None, "нет")."""
    first = layout["first_data_row"]
    head = " ".join(" ".join(cells) for no, cells in rows if no < first)
    years = re.findall(r"\b(20\d\d)\b", head)
    if years:
        return int(years[-1]), "шапка листа"
    date_col = col_index(layout["columns"]["date"])
    seen = {int(y) for no, cells in rows if no >= first
            for y in re.findall(r"\b(20\d\d)\b", cells[date_col])}
    return (seen.pop(), "даты листа") if len(seen) == 1 else (None, "нет")


for _sheet, _lay in {"Реестр_заявок_1": {"first_data_row": 7, "columns": {"date": "H"}},
                     "Реестр_заявок_2": {"first_data_row": 4, "columns": {"date": "G"}},
                     "Выгрузка_raw": {"first_data_row": 1, "columns": {"date": "G"}}}.items():
    print(f"год для дат без года на листе {_sheet}: {sheet_year(load_sheet(_sheet), _lay)}")

год для дат без года на листе Реестр_заявок_1: (2026, 'шапка листа')
год для дат без года на листе Реестр_заявок_2: (2026, 'шапка листа')
год для дат без года на листе Выгрузка_raw: (2025, 'даты листа')


In [17]:
# ═══════════════════════════ TODO 2 из 4 ═══════════════════════════
# normalize(rows, layout) -> list[dict]. Справка — над ячейкой.
#
# Что делает каждая функция в ячейке:
#   parse_wagons(text)     — достаёт количество вагонов из текста ячейки: "29" -> 29, "18 шт" -> 18,
#                            "18 ваг." -> 18; пустая ячейка -> None (число не восстанавливаем).
#   parse_date(text, year) — превращает текст даты в период (date_from, date_to), оба в формате ГГГГ-ММ-ДД.
#                            Точная дата ("03.08.2025", "2025-04-10") даёт одинаковые date_from и date_to.
#                            «начало / середина / конец месяца» даёт период 1–10 / 11–20 / 21–конец месяца.
#                            Если года в тексте нет, берётся year (его считает sheet_year); если и его нет — (None, None).
#   normalize(rows, layout) — главная функция. Идёт по строкам листа с first_data_row, пропускает пустые
#                            и «ИТОГО», чистит клиента / груз / род вагона / станции, вызывает parse_wagons
#                            и parse_date, выбрасывает дубли и возвращает список словарей-заявок.
#
# Каркас готов. Допишите 4 места с пометкой «ВАШ КОД» (всего 8 строк), выполните ячейку,
# затем «Диагностику TODO 2» и «Проверку TODO 2».
# ═══════════════════════════════════════════════════════════════════

def parse_wagons(text: str):
    """'29' -> 29, '18 шт' -> 18, '18 ваг.' -> 18, '' -> None."""
    m = re.search(r"\d+", text)
    return int(m.group()) if m else None


def parse_date(text: str, year):
    """Строка даты -> (date_from, date_to) в ISO, либо (None, None)."""
    t = text.strip().lower()
    if not t:
        return None, None
    m = re.fullmatch(r"(\d{2})\.(\d{2})\.(\d{4})", t)
    if m:
        iso = f"{m[3]}-{m[2]}-{m[1]}"
        return iso, iso
    m = re.fullmatch(r"(\d{4})-(\d{2})-(\d{2})", t)
    if m:
        return t, t
    m = re.fullmatch(r"(начало|середина|конец) (\w+)(?: (\d{4}))?", t)
    if m and m[2] in MONTHS:
        y = int(m[3]) if m[3] else year
        if y is None:
            return None, None
        month, (first_day, last_day) = MONTHS[m[2]], PERIODS[m[1]]
        last_day = last_day or calendar.monthrange(y, month)[1]   # «конец»: последний день месяца
        return f"{y}-{month:02d}-{first_day:02d}", f"{y}-{month:02d}-{last_day:02d}"
    raise ValueError(f"не знаю формат даты: {text!r}")


def normalize(rows, layout):
    year, _ = sheet_year(rows, layout)
    cols = {field: col_index(letter) for field, letter in layout["columns"].items()}
    out, seen = [], set()
    for no, cells in rows:
        if no < layout["first_data_row"] or not any(c.strip() for c in cells):
            continue
        if any(c.strip().upper() == "ИТОГО" for c in cells):   # ячейка целиком, а не подстрока
            continue
        get = lambda field: cells[cols[field]].strip()
        date_from, date_to = parse_date(get("date"), year)
        order = {"row": no, "client": " ".join(get("client").split()),
                 "cargo": get("cargo").lower(), "wagon_type": get("wagon_type").lower(),
                 "from_station": get("from_station"), "to_station": get("to_station"),
                 "wagons": parse_wagons(get("wagons")), "date_from": date_from, "date_to": date_to}
        key = tuple(v for k, v in order.items() if k != "row")   # дубль — все поля, кроме номера строки
        if key in seen:
            continue
        seen.add(key)
        out.append(order)
    return out

In [18]:
# @title Диагностика TODO 2: быстрые тесты и разбор строк
CASES_WAGONS = [("29", 29), ("18 шт", 18), ("18 ваг.", 18), ("", None)]
CASES_DATE = [("03.08.2025", None, ("2025-08-03", "2025-08-03")),
              ("2025-04-10", None, ("2025-04-10", "2025-04-10")),
              ("середина октября 2025", None, ("2025-10-11", "2025-10-20")),
              ("конец февраля 2025", None, ("2025-02-21", "2025-02-28")),
              ("начало марта", 2025, ("2025-03-01", "2025-03-10")),
              ("начало марта", None, (None, None))]


def quick_test_2() -> None:
    """Тесты parse_wagons и parse_date без модели и без Excel."""
    for text, want in CASES_WAGONS:
        got = parse_wagons(text)
        print("✅" if got == want else "❌", f"parse_wagons({text!r}) = {got!r}", "" if got == want else f"(нужно {want!r})")
    for text, year, want in CASES_DATE:
        try:
            got = parse_date(text, year)
        except Exception as e:  # noqa: BLE001
            got = f"{type(e).__name__}: {e}"
        print("✅" if got == want else "❌", f"parse_date({text!r}, {year}) = {got}", "" if got == want else f"(нужно {want})")


def try_row(sheet: str, row_no: int) -> None:
    """Сырая строка листа и то, что из неё получилось в normalize (нужны раскладки из TODO 1)."""
    raw = dict(load_sheet(sheet)).get(row_no)
    print("сырая строка:", raw)
    got = [o for o in normalize(load_sheet(sheet), LAYOUTS[sheet]) or [] if o["row"] == row_no]
    print("normalize   :", got[0] if got else "строка пропущена (пустая, «ИТОГО» или дубль)")


def show_dropped(sheet: str) -> None:
    """Какие строки данных normalize не вернул и по какой причине."""
    lay = LAYOUTS[sheet]
    kept = {o["row"] for o in normalize(load_sheet(sheet), lay) or []}
    for no, cells in load_sheet(sheet):
        if no < lay["first_data_row"] or no in kept:
            continue
        why = ("пустая" if not any(c.strip() for c in cells)
               else "ИТОГО" if any(c.strip().upper() == "ИТОГО" for c in cells) else "дубль")
        print(f"  строка {no}: {why}")


quick_test_2()

✅ parse_wagons('29') = 29 
✅ parse_wagons('18 шт') = 18 
✅ parse_wagons('18 ваг.') = 18 
✅ parse_wagons('') = None 
✅ parse_date('03.08.2025', None) = ('2025-08-03', '2025-08-03') 
✅ parse_date('2025-04-10', None) = ('2025-04-10', '2025-04-10') 
✅ parse_date('середина октября 2025', None) = ('2025-10-11', '2025-10-20') 
✅ parse_date('конец февраля 2025', None) = ('2025-02-21', '2025-02-28') 
✅ parse_date('начало марта', 2025) = ('2025-03-01', '2025-03-10') 
✅ parse_date('начало марта', None) = (None, None) 


## Проверка TODO 2

`normalize` запускается на трёх листах с раскладками из TODO 1. Если субагент ошибся колонкой, а
раскладку для проверки надо взять руками — впишите её: `LAYOUTS["лист"] = {"header_row": …,
"first_data_row": …, "columns": {…}}`.

In [19]:
# @title Нормализация листов и проверка TODO 2
def normalize_sheet(sheet: str) -> list[dict]:
    """normalize на одном листе с раскладкой из LAYOUTS. Готовый код."""
    try:
        out = normalize(load_sheet(sheet), LAYOUTS[sheet])
    except Exception as e:  # noqa: BLE001
        raise TodoNotDone(f"TODO 2 не выполнен: normalize упал на листе {sheet!r} "
                          f"({type(e).__name__}: {e}); допишите «ВАШ КОД» в ячейке TODO 2") from e
    need(out, 2, "normalize ничего не вернул — допишите функцию")
    return [{**o, "sheet": sheet} for o in out]


# Проверка TODO 2
try:
    need(LAYOUTS or None, 2, "нет раскладок: сначала «Проверка TODO 1»")
    for _sheet in SHEETS:
        _year = sheet_year(load_sheet(_sheet), LAYOUTS[_sheet])
        _n = len(normalize_sheet(_sheet))
        print(f"{_sheet}: {_n} заявок · год для дат без года {_year[0]} ({_year[1]})")
    print()
    check(2, 3)
except TodoNotDone as e:
    explain_todo(e)

Реестр_заявок_1: 45 заявок · год для дат без года 2026 (шапка листа)
Реестр_заявок_2: 40 заявок · год для дат без года 2026 (шапка листа)
Выгрузка_raw: 35 заявок · год для дат без года 2025 (даты листа)

✅ 2. TODO 2 · 120 уникальных заявок: 120 уникальных заявок (45 + 40 + 35), «ИТОГО» нет, «Магнитогорск-Грузовой» на месте
✅ 3. TODO 2 · даты и количество: даты ISO, количество int или None, год для дат без года выбран по правилу
Итого: 2 из 2


---
# TODO 3 · Справочник важнее догадки

## Зачем

В файле четыре станции с опечатками: перестановка соседних букв. Модель такие станции «исправит»
по догадке — а завтра исправит иначе. Станцию определяет **справочник**: `stations.json`, 120
станций. Догадку заменяет формальное правило.

## Что написать: `match_station(name, known)`

`known` — список названий из справочника. Верните название **из справочника** или `None`:

1. сравните без внешних пробелов и без учёта регистра; **точное совпадение** — вернуть название из справочника;
2. иначе возьмите ближайшее по `difflib.SequenceMatcher(None, a, b).ratio()` (сравнивайте в нижнем регистре);
3. вернуть его, только если `ratio` лучшего **не ниже 0.60** и **отрыв от второго лучшего не меньше 0.10**
   — у коротких названий соседи близко, и «исправить» верное слово нельзя;
4. иначе `None` — станция уйдёт в отчёт как неизвестная.

**Что дано, что дописать:** готов шаг 1 (точное совпадение). Допишите шаги 2–4: список кандидатов с оценками, два лучших, порог 0.60 и отрыв 0.10 — 7–8 строк.

In [20]:
# ═══════════════════════════ TODO 3 из 4 ═══════════════════════════
# match_station(name, known) -> str | None. Справка — над ячейкой.
# Шаг 1 готов. Допишите шаги 2–4 (7–8 строк), выполните ячейку,
# затем «Диагностику TODO 3», «Сборку базы» и «Проверку TODO 3».
# ═══════════════════════════════════════════════════════════════════
import difflib


def match_station(name: str, known: list[str]):
    wanted = name.strip().lower()
    for station in known:                      # шаг 1: точное совпадение без регистра и внешних пробелов
        if station.lower() == wanted:
            return station
    scored = []                                # шаг 2: оценки по всем станциям
    for s in known:
        scored.append((difflib.SequenceMatcher(None, wanted, s.lower()).ratio(), s))
    scored.sort(reverse=True)
    best, second = scored[0], (scored[1] if len(scored) > 1 else (0.0, ""))   # шаг 3: два лучших
    if best[0] >= 0.60 and best[0] - second[0] >= 0.10:                        # шаг 4: порог и отрыв
        return best[1]
    return None

In [21]:
# @title Диагностика TODO 3: быстрые тесты
def _swap(s: str) -> str:
    return s[:2] + s[3] + s[2] + s[4:]


_a, _b = STATION_NAMES[5], STATION_NAMES[40]
CASES_STATION = [("точное название", _a, _a), ("с пробелами и в верхнем регистре", f"  {_a.upper()} ", _a),
                 ("опечатка (две соседние буквы)", _swap(_b), _b), ("такой станции нет", "Нетакаястанцияxyz", None)]
for _what, _inp, _want in CASES_STATION:
    _got = match_station(_inp, STATION_NAMES)
    print("✅" if _got == _want else "❌", f"{_what}: {_inp!r} -> {_got!r}", "" if _got == _want else f"(нужно {_want!r})")

✅ точное название: 'Затишье-Северная' -> 'Затишье-Северная' 
✅ с пробелами и в верхнем регистре: '  ЗАТИШЬЕ-СЕВЕРНАЯ ' -> 'Затишье-Северная' 
✅ опечатка (две соседние буквы): 'Соеснки-Промышленная' -> 'Сосенки-Промышленная' 
✅ такой станции нет: 'Нетакаястанцияxyz' -> None 


---
## 8. Сборка базы — готовая

Нормализованные заявки пишутся в отдельную SQLite-базу в памяти — таблицу `normalized_orders`.
Станции проходят через ваш `match_station`. Отчёт ниже: сколько станций исправлено и какие не нашлись.

In [22]:
# @title Сборка базы
import sqlite3


def build_database() -> sqlite3.Connection:
    """normalize -> match_station -> SQLite :memory:. Готовый код."""
    FINAL_ORDERS.clear(); FIXED_STATIONS.clear(); UNKNOWN_STATIONS.clear(); FINAL_BY_KEY.clear()
    for sheet in SHEETS:
        for o in normalize_sheet(sheet):
            for field in ("from_station", "to_station"):
                found = match_station(o[field], STATION_NAMES)
                if found is None:
                    UNKNOWN_STATIONS.append(o[field])
                elif found != o[field]:
                    FIXED_STATIONS.append((o[field], found))
                    o[field] = found
            FINAL_ORDERS.append(o)
            FINAL_BY_KEY[f"{sheet}:{o['row']}"] = o
    db = sqlite3.connect(":memory:", check_same_thread=False)
    db.execute("""CREATE TABLE normalized_orders (sheet TEXT, row INTEGER, client TEXT, cargo TEXT,
                  wagon_type TEXT, from_station TEXT, to_station TEXT, wagons INTEGER,
                  date_from TEXT, date_to TEXT)""")
    db.executemany("INSERT INTO normalized_orders VALUES (?,?,?,?,?,?,?,?,?,?)",
                   [(o["sheet"], o["row"], o["client"], o["cargo"], o["wagon_type"], o["from_station"],
                     o["to_station"], o["wagons"], o["date_from"], o["date_to"]) for o in FINAL_ORDERS])
    db.commit()
    return db


def show_build_report() -> None:
    print(f"заявок в базе: {len(FINAL_ORDERS)}")
    print(f"исправлено станций: {len(FIXED_STATIONS)}", *[f"\n  {a} -> {b}" for a, b in FIXED_STATIONS])
    print(f"не найдено в справочнике: {len(UNKNOWN_STATIONS)}", *[f"\n  {a}" for a in UNKNOWN_STATIONS])


print("Сборка готова.")

Сборка готова.


## Проверка TODO 3

In [23]:
# Проверка TODO 3
try:
    need(LAYOUTS or None, 3, "нет раскладок: сначала «Проверка TODO 1»")
    need(match_station("Инская", STATION_NAMES), 3, "match_station не вернул станцию даже на точном совпадении")
    DB = build_database()
    show_build_report()
    print()
    check(4)
except TodoNotDone as e:
    explain_todo(e)

заявок в базе: 120
исправлено станций: 4 
  Кемерово-Сотрировочное -> Кемерово-Сортировочное 
  Сосенки-рГузовая -> Сосенки-Грузовая 
  Уаф -> Уфа 
  Соляная-Прмоышленная -> Соляная-Промышленная
не найдено в справочнике: 0

✅ 4. TODO 3 · станции по справочнику: 4 опечатки исправлены, неизвестных 0, верные станции не тронуты
Итого: 1 из 1


---
## 9. `query_orders` и контрольный вопрос — готовые

`query_orders` — инструмент агента над таблицей `normalized_orders`. Его аргументы — **параметры**
(груз, клиент, станции, период), а запрос собирает код: модель SQL не пишет. Один и тот же вызов
всегда даёт один и тот же ответ, а ошибка видна в аргументах.

Контрольный вопрос: «Сколько вагонов под уголь заявлено на октябрь 2025?» Период считается так:
заявка попадает в октябрь, если её интервал `[date_from, date_to]` пересекает месяц.

In [24]:
# @title query_orders и контрольный вопрос
ISO = re.compile(r"\d{4}-\d{2}-\d{2}")


def query_orders(cargo=None, client=None, from_station=None, to_station=None,
                 period_from=None, period_to=None) -> str:
    """Параметры -> запрос к normalized_orders. SQL собирает этот код, не модель."""
    where, args = [], []
    for column, value in (("cargo", cargo), ("client", client), ("from_station", from_station),
                          ("to_station", to_station)):
        if value:
            where.append(f"{column} = ?")
            args.append(value)
    for name, value in (("period_from", period_from), ("period_to", period_to)):
        if value and not ISO.fullmatch(str(value)):
            return f"Ошибка: {name} должен быть датой ГГГГ-ММ-ДД, а пришло {value!r}."
    if period_to:        # заявка попадает в период, если её интервал пересекает период
        where.append("date_from <= ?")
        args.append(period_to)
    if period_from:
        where.append("date_to >= ?")
        args.append(period_from)
    sql = ("SELECT COUNT(*), COALESCE(SUM(wagons), 0) FROM normalized_orders"
           + (" WHERE " + " AND ".join(where) if where else ""))
    orders, wagons = DB.execute(sql, args).fetchone()
    return json.dumps({"orders": orders, "wagons": wagons}, ensure_ascii=False)


def _cargo_values() -> list[str]:
    return sorted({o["cargo"] for o in FINAL_ORDERS})


def query_tool() -> list[dict]:
    return [{"type": "function", "function": {
        "name": "query_orders",
        "description": ("Сколько заявок и вагонов в нормализованных заявках. Все параметры необязательны и "
                        "сужают выборку. Период — две даты ГГГГ-ММ-ДД: в него попадает заявка, чей "
                        "интервал подачи пересекает период. Заявки без даты в период не попадают."),
        "parameters": {"type": "object", "properties": {
            "cargo": {"type": "string", "enum": _cargo_values(), "description": "род груза"},
            "client": {"type": "string", "description": "клиент, как в данных"},
            "from_station": {"type": "string", "description": "станция отправления"},
            "to_station": {"type": "string", "description": "станция назначения"},
            "period_from": {"type": "string", "description": "начало периода, ГГГГ-ММ-ДД"},
            "period_to": {"type": "string", "description": "конец периода, ГГГГ-ММ-ДД"}}}}}]


def query_call(name: str, args: dict) -> str:
    if name != "query_orders":
        return f"Ошибка: нет инструмента «{name}». Есть только query_orders."
    try:
        return query_orders(**args)
    except TypeError as e:
        return f"Ошибка в аргументах query_orders: {e}"


QUERY_SYSTEM = f"""\
Ты — помощник по заявкам на подачу вагонов. Отвечаешь только через инструмент query_orders, сам чисел
не знаешь. Сегодня {TODAY}. Период в вопросе переведи в две даты и укажи их в period_from и period_to.
В ответе назови период, груз и единицы (вагоны)."""
Q_CONTROL = "Сколько вагонов под уголь заявлено на октябрь 2025?"


def run_control() -> dict:
    CONTROL_RUN.clear()
    CONTROL_RUN.update(run_agent(Q_CONTROL, system=QUERY_SYSTEM, tools=query_tool(), call=query_call,
                                 max_steps=5))
    return CONTROL_RUN


print("query_orders готов.")

query_orders готов.


## Контрольный вопрос

Агент отвечает на вопрос через `query_orders`. Этот прогон сохраняется как `CONTROL_RUN` — по нему
проверка 6 смотрит, какие аргументы агент отправил и какое число назвал.

In [25]:
# Контрольный вопрос
try:
    need(FINAL_ORDERS or None, 3, "сначала «Проверка TODO 3» — она собирает базу")
    run_control()
    print("\nответ:", CONTROL_RUN["answer"])
    print()
    check(6)
except TodoNotDone as e:
    explain_todo(e)

  шаг 1 · в модель ушло 635 ток. контекста
          query_orders({"cargo": "уголь каменный", "period_from": "2025-10-01", "period_to": "2025-10-31"})
            -> {"orders": 4, "wagons": 75}
  шаг 2 · в модель ушло 745 ток. контекста

ответ: За октябрь 2025 (01.10.2025–31.10.2025) по грузу «уголь каменный» заявлено 4 заявки, всего 75 вагонов.

✅ 6. Контрольный вопрос через query_orders: агент ответил через query_orders, число 75 совпало, SQL в аргументах нет
Итого: 1 из 1


---
# TODO 4 · Ограничители с самого начала

## Зачем

Оркестратор, который может звать субагентов без конца, — это счёт за токены и бесконечный цикл. В
исследовательской системе AOrchestra у оркестратора не больше 10 попыток, у субагента — не больше 50
шагов. Ваши ограничители проще, но принцип тот же: **предел задан кодом, а отказ — не исключение, а
понятный текст**, который оркестратор прочитает и перестроит план.

## Что написать: `check_limits(state, sheet)`

`state` — словарь `{"agents": сколько субагентов уже запущено, "completion_tokens": сколько токенов
ответа они потратили, "depth": глубина вызова (0 — зовёт оркестратор, 1 — зовёт субагент)}`.
Верните `None`, если запускать можно, и **текст ошибки**, если нельзя:

| Условие | Константа |
|---|---|
| `agents` уже не меньше `MAX_AGENTS` (3) — четвёртый вызов отказывается | `MAX_AGENTS` |
| `depth` больше `MAX_NESTED_SPAWN` (0) — субагент не порождает субагентов | `MAX_NESTED_SPAWN` |
| `completion_tokens` уже не меньше `MAX_TOTAL_COMPLETION_TOKENS` (2048) | `MAX_TOTAL_COMPLETION_TOKENS` |

Третье ограничение — в самом `make_subagent` (TODO 1): `spawn_agent` в `tools` субагента нет.
Исключения не бросайте: текст ошибки должен быть понятен оркестратору, и в нём должно быть сказано,
что делать дальше.

**Что дано, что дописать:** в ячейке только комментарии-шаги. Допишите три проверки с текстами отказов — 6–7 строк.

In [26]:
# ═══════════════════════════ TODO 4 из 4 ═══════════════════════════
# check_limits(state, sheet) -> None | str. Справка — над ячейкой.
# Допишите три проверки (6–7 строк), выполните ячейку,
# затем «Диагностику TODO 4» и «Проверку TODO 4».
# ═══════════════════════════════════════════════════════════════════

def check_limits(state: dict, sheet: str):
    if state["agents"] >= MAX_AGENTS:
        return (f"Отказ: уже запущено {state['agents']} субагентов из {MAX_AGENTS}. "
                "Новых не запускай — закончи ответом по уже полученным раскладкам.")
    if state["depth"] > MAX_NESTED_SPAWN:
        return ("Отказ: субагент не может запускать других субагентов. "
                "Верни раскладку своего листа и закончи.")
    if state["completion_tokens"] >= MAX_TOTAL_COMPLETION_TOKENS:
        return (f"Отказ: израсходован лимит токенов ({MAX_TOTAL_COMPLETION_TOKENS}). "
                "Новых субагентов не запускай — закончи ответом по уже полученным раскладкам.")
    return None

In [27]:
# @title Диагностика TODO 4: что отвечает check_limits
_fresh = {"agents": 0, "completion_tokens": 0, "depth": 0}
for _what, _state in [("свежее состояние", _fresh), ("агентов уже 3", {**_fresh, "agents": MAX_AGENTS}),
                      ("вызов из субагента", {**_fresh, "depth": 1}),
                      ("токены исчерпаны", {**_fresh, "completion_tokens": MAX_TOTAL_COMPLETION_TOKENS})]:
    print(f"{_what:<20} -> {check_limits(dict(_state), SHEETS[0])!r}")

свежее состояние     -> None
агентов уже 3        -> 'Отказ: уже запущено 3 субагентов из 3. Новых не запускай — закончи ответом по уже полученным раскладкам.'
вызов из субагента   -> 'Отказ: субагент не может запускать других субагентов. Верни раскладку своего листа и закончи.'
токены исчерпаны     -> 'Отказ: израсходован лимит токенов (2048). Новых субагентов не запускай — закончи ответом по уже полученным раскладкам.'


## Проверка TODO 4

Состояние лимитов выставляется так, будто три субагента уже отработали, — и `spawn_agent` зовут в
четвёртый раз. Модель при этом не вызывается: отказ приходит раньше.

In [28]:
# Проверка TODO 4
try:
    need(check_limits({"agents": MAX_AGENTS, "completion_tokens": 0, "depth": 0}, SHEETS[0]), 4,
         "check_limits ничего не вернул на четвёртом вызове — допишите функцию")
    saved = dict(SPAWN_STATE)
    SPAWN_STATE.update(agents=MAX_AGENTS, completion_tokens=0, depth=0)
    LIMIT_REPLY = spawn_agent(SHEETS[0])
    SPAWN_STATE.update(saved)
    print("ответ spawn_agent на четвёртый вызов:\n ", LIMIT_REPLY)
    print()
    check(5)
except TodoNotDone as e:
    explain_todo(e)

ответ spawn_agent на четвёртый вызов:
  Отказ: уже запущено 3 субагентов из 3. Новых не запускай — закончи ответом по уже полученным раскладкам.

✅ 5. TODO 4 · ограничители: четвёртый вызов получил текст ошибки, у субагентов нет spawn_agent
Итого: 1 из 1


---
### Контрольная точка 2 · сколько стоили агенты

Сравним два режима на одних данных и одном вопросе:

1. **Один агент:** все три листа целиком уходят в модель текстом, инструментов нет.
2. **Оркестратор:** субагенты видят по десять строк, строки разбирает код, а ответ даёт `query_orders`.

Смотрим не на впечатление, а на токены и на то, верен ли ответ.

In [29]:
# @title Один агент на все три листа и таблица токенов
def all_sheets_text() -> str:
    parts = []
    for sheet in SHEETS:
        parts.append(f"=== лист «{sheet}» ===")
        parts += [f"{no}\t" + "\t".join(cells) for no, cells in load_sheet(sheet)]
    return "\n".join(parts)


def run_single_agent() -> dict:
    """Один агент: три листа целиком в контексте, инструментов нет."""
    system = ("Ты отвечаешь на вопросы по заявкам на подачу вагонов. Данные — три листа Excel ниже, "
              "как есть. Верни число и коротко назови, как считал.\n\n" + all_sheets_text())
    SINGLE_RUN.clear()
    SINGLE_RUN.update(run_agent(Q_CONTROL, system=system, max_steps=1, verbose=False))
    return SINGLE_RUN


def token_table():
    import pandas as pd

    def stated_wagons(text):
        """Число перед словом «ваг…» — то, что агент назвал ответом (а не любое число в тексте)."""
        m = re.search(r"(\d[\d\s\u00a0\u202f]*)\s*ваг", str(text))
        return int(re.sub(r"\D", "", m.group(1))) if m else None

    def line(name, runs, answer=None, strict=False):
        runs = runs if isinstance(runs, list) else [runs]
        return {"режим": name, "токены на вход": total(runs, "prompt"),
                "токены на выход": total(runs, "completion"),
                "верный ответ": "—" if answer is None else ("да" if is_right(answer, strict) else "нет")}

    def is_right(answer, strict):
        want = PUBLIC["control"]["wagons"]
        if strict and stated_wagons(answer) is not None:
            return stated_wagons(answer) == want
        return has_number(answer, want)

    rows = []
    if SINGLE_RUN:
        rows.append(line("Один агент, три листа целиком", SINGLE_RUN, SINGLE_RUN["answer"], strict=True))
    if ORCH_RUN:
        rows.append(line("Оркестратор", ORCH_RUN))
        rows.append(line("Субагенты (по десять строк)", {"usage": [x for a in AGENTS_LOG for x in a["usage"]]}))
    if CONTROL_RUN:
        rows.append(line("query_orders — ответ", CONTROL_RUN, CONTROL_RUN["answer"]))
    if ORCH_RUN and CONTROL_RUN:
        rows.append(line("Оркестратор — итого", [ORCH_RUN, {"usage": [x for a in AGENTS_LOG for x in a["usage"]]}, CONTROL_RUN], CONTROL_RUN["answer"]))
    if not rows:
        return "Таблица заполнится после ячеек «Проверка TODO 1» и «Контрольный вопрос»."
    return pd.DataFrame(rows).set_index("режим")


print("Сравнение готово.")

Сравнение готово.


In [30]:
# Контрольная точка 2
if CONTROL_RUN:
    run_single_agent()
    print("ответ одного агента:", SINGLE_RUN["answer"][:200])
    print()
else:
    print("Сначала выполните «Контрольный вопрос» — таблица сравнивает его с одним агентом.")
token_table()

ответ одного агента: **0**

Считал так: просмотрел все три листа, искал строки, где груз — уголь каменный (включая «УГОЛЬ КАМЕННЫЙ») и дата подачи приходится на октябрь 2025. В «Реестре_заявок_1» уголь с октябрём не встре



,токены на вход,токены на выход,верный ответ
режим,,,
"Один агент, три листа целиком",6249,396,нет
Оркестратор,1302,456,—
Субагенты (по десять строк),6118,276,—
query_orders — ответ,1380,132,да
Оркестратор — итого,8800,864,да


---
## `check()` — критерий сдачи

Шесть проверок: по одной на каждое задание и контрольный вопрос — те же, что печатали ячейки
«Проверка». На занятии минимум — **1–3**, работа сдана при **6/6**.

In [31]:
# @title Итоговая проверка
check()

✅ 1. TODO 1 · раскладки листов: три раскладки листов совпали с эталоном (шапка в строке 6 / 3 / нет)
✅ 2. TODO 2 · 120 уникальных заявок: 120 уникальных заявок (45 + 40 + 35), «ИТОГО» нет, «Магнитогорск-Грузовой» на месте
✅ 3. TODO 2 · даты и количество: даты ISO, количество int или None, год для дат без года выбран по правилу
✅ 4. TODO 3 · станции по справочнику: 4 опечатки исправлены, неизвестных 0, верные станции не тронуты
✅ 5. TODO 4 · ограничители: четвёртый вызов получил текст ошибки, у субагентов нет spawn_agent
✅ 6. Контрольный вопрос через query_orders: агент ответил через query_orders, число 75 совпало, SQL в аргументах нет
Итого: 6 из 6


6

---
## Домашнее задание

1. **Довести `check()` до 6/6.** На минимальной траектории TODO 3 и 4 — дома.
2. **Отчёт о качестве загрузки:** сколько строк отброшено и почему — по каждому листу. Строк с
   данными в листе: `len([r for r in load_sheet(лист) if r[0] >= first_data_row and any(r[1])])`;
   в базе: `FINAL_ORDERS`. Разница — пустые строки, «ИТОГО», дубли; разберите по причинам.
3. **По желанию:** замените раздачу листов через модель на обычный скрипт, который сам вызывает
   `spawn_agent` для каждого листа, и сравните токены оркестратора. Когда обход задан данными, его
   ведёт код — это и есть вывод лекции.

Рекурсию (субагент зовёт субагентов) в домашнем не делаем: она запрещена в TODO 4.

In [32]:
import pandas as pd

report = []
for sheet in SHEETS:
    first = LAYOUTS[sheet]["first_data_row"]
    body = [r for r in load_sheet(sheet) if r[0] >= first]
    empty = [r for r in body if not any(c.strip() for c in r[1])]
    totals = [r for r in body if any(c.strip().upper() == "ИТОГО" for c in r[1])]
    kept = [o for o in FINAL_ORDERS if o["sheet"] == sheet]
    report.append({"лист": sheet, "строк после шапки": len(body), "пустых": len(empty),
                   "«ИТОГО»": len(totals),
                   "дублей": len(body) - len(empty) - len(totals) - len(kept),
                   "в базе": len(kept)})
pd.DataFrame(report).set_index("лист")

,строк после шапки,пустых,«ИТОГО»,дублей,в базе
лист,,,,,
Реестр_заявок_1,49,1,1,2,45
Реестр_заявок_2,43,1,1,1,40
Выгрузка_raw,36,0,0,1,35
